In [1]:
!git clone https://github.com/bhujbalatharva252-sketch/Recommendation-Systems-benchmarking

Cloning into 'Recommendation-Systems-benchmarking'...
remote: Enumerating objects: 62, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 62 (delta 2), reused 5 (delta 1), pack-reused 56 (from 1)
Receiving objects: 100% (62/62), 41.39 MiB | 10.38 MiB/s, done.
Resolving deltas: 100% (17/17), done.
Updating files: 100% (33/33), done.


In [8]:
%cd /content/Recommendation-Systems-benchmarking

/content/Recommendation-Systems-benchmarking


In [11]:
import pandas as pd

In [15]:
import pandas as pd
import numpy as np

# Load processed datasets
train = pd.read_csv("data/processed/train.csv")
test = pd.read_csv("data/processed/test.csv")
movies = pd.read_csv("data/raw/movies.dat",
    sep="::",
    engine="python",
    names=["MovieID", "Title", "Genres"],
    encoding="latin-1")


print(train.shape)
print(test.shape)
#print(movies.shape)

(993571, 10)
(6040, 10)


# Candidate Set Generation

In [16]:
import numpy as np

# Number of negative samples of each user (99 negative + 1 true test = 100 candidates)
N_NEGATIVES= 99

# Get all available movie IDs
all_movie_ids= movies["MovieID"].unique()

# Each user gets 1 positive item (item from test set), 99 negative items (Movies the user has not interacted with )
def build_candidate_sets(test_df, train_df, all_movie_ids, n_negatives=N_NEGATIVES, seed=42):
    rng = np.random.default_rng(seed)       # reproducible sampling
    candidates = {}

    # Store movies each user has already interacted with in training data
    user_seen = train_df.groupby("UserID")["MovieID"].apply(set)

    # Create candidate set for each test user
    for _, row in test_df.iterrows():

        # Get user ID and actual test movie (positive item)
        user_id = row["UserID"]
        test_item = row["MovieID"]

        # Include test item to prevent sampling it as a negative
        seen= user_seen.get(user_id, set()) | {test_item}

        # Select movies that the user has never interacted with
        neg_pool= np.setdiff1d(all_movie_ids, list(seen))

        # Randomly select negative samples
        n= min(n_negatives, len(neg_pool))
        negatives= rng.choice(neg_pool, size=n, replace=False)

        # Combine negative items with the true test item
        items = list(negatives) + [test_item]
        rng.shuffle(items)  # avoid test item always in same position
        candidates[user_id]= {"test_item": test_item, "items": items}

    return candidates

# Generate final candidate sets
candidate_sets = build_candidate_sets(test, train, all_movie_ids)
print(f"Built candidate sets for {len(candidate_sets)} users "
      f"({N_NEGATIVES} negatives + 1 test item each).")

Built candidate sets for 6040 users (99 negatives + 1 test item each).


In [24]:
# Example
# Select one user
user_id= list(candidate_sets.keys())[0]

# Display candidate set
candidate_sets[1]

{'test_item': 48,
 'items': [np.int64(705),
  np.int64(2756),
  np.int64(3329),
  np.int64(3561),
  np.int64(1783),
  np.int64(502),
  np.int64(1694),
  np.int64(1444),
  np.int64(2720),
  np.int64(3007),
  np.int64(2493),
  np.int64(1632),
  np.int64(1988),
  np.int64(1387),
  np.int64(3213),
  np.int64(3793),
  np.int64(3706),
  np.int64(2176),
  np.int64(2786),
  np.int64(1236),
  np.int64(2194),
  np.int64(3620),
  np.int64(3063),
  np.int64(3145),
  np.int64(3773),
  np.int64(354),
  np.int64(1862),
  np.int64(2549),
  np.int64(360),
  np.int64(3051),
  np.int64(3355),
  np.int64(3053),
  np.int64(341),
  np.int64(1770),
  np.int64(2803),
  np.int64(1462),
  np.int64(627),
  np.int64(1777),
  np.int64(2156),
  np.int64(3465),
  np.int64(1867),
  np.int64(3558),
  np.int64(2702),
  np.int64(946),
  np.int64(3061),
  np.int64(597),
  np.int64(890),
  np.int64(776),
  np.int64(2725),
  np.int64(1913),
  np.int64(1746),
  np.int64(3197),
  np.int64(3270),
  np.int64(2991),
  np.int64(

# Save Candidate Set

In [30]:
import pickle

# Store candidate set as pickle file
with open("candidate_sets_100.pkl", "wb") as f:
    pickle.dump(candidate_sets, f)

In [31]:
from google.colab import files
files.download("candidate_sets_100.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>